In [1]:
import yaml
import requests
from datetime import datetime, timedelta
from fastapi import FastAPI, HTTPException
import joblib
import numpy as np
from pydantic_settings import BaseSettings

In [3]:
# 加载配置文件
with open(r"D:\Aa中工互联\工作安排\英格索兰\config.yml", "r", encoding="utf-8") as f:  
    config = yaml.safe_load(f)

# 从配置中提取参数
API_URL = config["api_config"]["url"]
API_PARAMS = config["api_config"]["params"]
VARIABLE_MAPPING = config["variable_mapping"]
MODEL_PATH = config["model_config"]["model_path"]
TIME_FEATURES = config["time_features"]
APP_INFO = config["app_info"]

In [47]:
API_PARAMS["names"]

['DLDZ_DQ200_SYSTEM_PI05.PV',
 'DLDZ_AVS_SYSTEM_PI05.PV',
 'DLDZ_DQ200_LLJ01_FQ01.PV',
 'DLDZ_AVS_LLJ01_FQ01.PV']

In [48]:
# 处理 names 参数：将列表转为逗号分隔的字符串
if "names" in API_PARAMS and isinstance(API_PARAMS["names"], list):
    API_PARAMS["names"] = ",".join(API_PARAMS["names"])

In [49]:
API_PARAMS["names"]

'DLDZ_DQ200_SYSTEM_PI05.PV,DLDZ_AVS_SYSTEM_PI05.PV,DLDZ_DQ200_LLJ01_FQ01.PV,DLDZ_AVS_LLJ01_FQ01.PV'

In [5]:
# 初始化FastAPI应用
app = FastAPI()

# 加载模型
model = joblib.load(r'D:\Aa中工互联\工作安排\英格索兰\xgb_reg_model.pkl')

In [50]:
"""从API获取实时数据（使用YAML配置参数）"""
# 时间范围计算
end_time = datetime.now()
start_time = end_time - timedelta(hours=1)

# 构建动态参数
params = {
    "startTime": start_time.isoformat(timespec='milliseconds'),
    "endTime": end_time.isoformat(timespec='milliseconds'),
    **API_PARAMS  # 合并固定参数
}
params

{'startTime': '2025-05-21T09:33:53.086',
 'endTime': '2025-05-21T10:33:53.086',
 'interval': 60000,
 'valueonly': 0,
 'decimal': 2,
 'names': 'DLDZ_DQ200_SYSTEM_PI05.PV,DLDZ_AVS_SYSTEM_PI05.PV,DLDZ_DQ200_LLJ01_FQ01.PV,DLDZ_AVS_LLJ01_FQ01.PV'}

In [38]:
params ={'startTime': '2025-05-21T08:46:09.442',
 'endTime': '2025-05-21T09:46:09.442',
 'interval': 600000,
 'valueonly': 0,
 'decimal': 2,
 'names': ['DLDZ_DQ200_SYSTEM_PI05.PV','DLDZ_AVS_SYSTEM_PI05.PV']}

In [43]:
params = {
    'startTime': '2025-05-21T08:46:09.442',
    'endTime': '2025-05-21T09:46:09.442',
    'interval': 600000,
    'valueonly': 0,
    'decimal': 2,
    'names': ','.join(['DLDZ_DQ200_SYSTEM_PI05.PV', 'DLDZ_AVS_SYSTEM_PI05.PV'])  # 转换为逗号分隔的字符串
}

In [51]:
# 发送请求
try:
    response = requests.get(API_URL, params=params, timeout=10)
    response.raise_for_status()
except Exception as e:
    raise Exception(f"API请求失败: {str(e)}")

In [40]:
# 创建请求对象并准备（不发送请求）
req = requests.Request('GET', API_URL, params=params)
prepared_req = req.prepare()

# 输出最终的完整 URL
print("完整 URL:", prepared_req.url)

完整 URL: http://8.130.25.118:8000/api/hisdata?startTime=2025-05-21T08%3A46%3A09.442&endTime=2025-05-21T09%3A46%3A09.442&interval=600000&valueonly=0&decimal=2&names=DLDZ_DQ200_SYSTEM_PI05.PV&names=DLDZ_AVS_SYSTEM_PI05.PV


In [52]:
# 解析响应
data = response.json()
if data['code'] != 0:
    raise ValueError(f"API返回错误码: {data['code']}")

In [53]:
len(data['items'])

4

In [54]:
data

{'code': 0,
 'items': [{'name': 'DLDZ_DQ200_SYSTEM_PI05.PV',
   'vals': [{'time': '2025-05-21T09:33:53.086', 'val': 5.91},
    {'time': '2025-05-21T09:34:53.086', 'val': 5.92},
    {'time': '2025-05-21T09:35:53.086', 'val': 5.93},
    {'time': '2025-05-21T09:36:53.086', 'val': 5.92},
    {'time': '2025-05-21T09:37:53.086', 'val': 5.94},
    {'time': '2025-05-21T09:38:53.086', 'val': 5.94},
    {'time': '2025-05-21T09:39:53.086', 'val': 5.89},
    {'time': '2025-05-21T09:40:53.086', 'val': 5.87},
    {'time': '2025-05-21T09:41:53.086', 'val': 5.88},
    {'time': '2025-05-21T09:42:53.086', 'val': 5.89},
    {'time': '2025-05-21T09:43:53.086', 'val': 5.93},
    {'time': '2025-05-21T09:44:53.086', 'val': 5.93},
    {'time': '2025-05-21T09:45:53.086', 'val': 5.94},
    {'time': '2025-05-21T09:46:53.086', 'val': 5.96},
    {'time': '2025-05-21T09:47:53.086', 'val': 5.91},
    {'time': '2025-05-21T09:48:53.086', 'val': 5.94},
    {'time': '2025-05-21T09:49:53.086', 'val': 5.88},
    {'time': 

In [55]:
# 提取变量数据
input_data = {}
for var_api, var_model in VARIABLE_MAPPING.items():
    item = next((item for item in data['items'] if item['name'] == var_api), None)
    if not item:
        raise ValueError(f"未找到变量: {var_api}")
    if not item['vals']:
        raise ValueError(f"变量 {var_api} 数据为空")
    input_data[var_model] = item['vals'][-1]['val']

http://8.130.25.118:8000/api/userlogin/?user=user1&pass=123456

In [ ]:
http://8.130.25.118/api/realdata?tags=A1,A2,A3&pars=PV,DESC&token=12345678&v
alueonly=1

http://8.130.25.118/api/realdata?tags=DLDZ_DQ200_SYSTEM_P105&pars=PV

DLDZ_DQ200_SYSTEM_PI05 = DQ200干系统压力
DLDZ_AVS_SYSTEM_PI05 = AVS干系统压力
DLDZ_DQ200_SYSTEM_FQ01 = DQ200系统累积流量
DLDZ_AVS_SYSTEM_FQ01 = AVS系统累积流量

In [2]:
# 计算时间范围（当前时间的前一小时至当前时间）
end_time = datetime.now()
start_time = end_time - timedelta(hours=1)

# 构建API请求参数
params = {
    "startTime": start_time.isoformat(timespec='milliseconds'),
    "endTime": end_time.isoformat(timespec='milliseconds'),
    "interval": 600000,  # 10分钟的毫秒数
    "valueonly": 0,
    "decimal": 2,
    "names": "DLDZ_DQ200_SYSTEM_PI05.PV,DLDZ_AVS_SYSTEM_PI05.PV,DLDZ_DQ200_LLJ01_FQ01.PV,DLDZ_AVS_LLJ01_FQ01.PV"
}

In [ ]:
# 发送GET请求
try:
    response = requests.get(
        "http://8.130.25.118:8000/api/hisdata",
        params=params,
        timeout=10  # 设置超时时间
    )
    response.raise_for_status()  # 检查HTTP状态码
except Exception as e:
    raise Exception(f"API请求失败: {str(e)}")

In [4]:
# 解析响应数据
data = response.json()
if data['code'] != 1:
    raise ValueError(f"API返回错误码: {data['code']}")

ValueError: API返回错误码: 0

In [10]:
# 变量映射关系
variable_mapping = {
    "DLDZ_DQ200_SYSTEM_PI05.PV": "DQ200系统压力",
    "DLDZ_AVS_SYSTEM_PI05.PV": "AVS系统压力",
    "DLDZ_DQ200_LLJ01_FQ01.PV": "DQ200累积流量",
    "DLDZ_AVS_LLJ01_FQ01.PV": "AVS累积流量"
}

input_data = {}

# 提取各变量的最新值
for var_api, var_model in variable_mapping.items():
    item = next((item for item in data['items'] if item['name'] == var_api), None)
    if not item:
        raise ValueError(f"未找到变量: {var_api}")
    if not item['vals']:
        raise ValueError(f"变量 {var_api} 数据为空")
    input_data[var_model] = item['vals'][-1]['val']

# 计算系统压力的差分值（最后两个数据点差值取整）
def calculate_diff(var_name):
    item = next(item for item in data['items'] if item['name'] == var_name)
    vals = item['vals']
    if len(vals) >= 2:
        return round(vals[-1]['val'] - vals[-2]['val'])
    return 0

input_data["DQ200差分值"] = calculate_diff("DLDZ_DQ200_LLJ01_FQ01.PV")
input_data["AVS差分值"] = calculate_diff("DLDZ_AVS_LLJ01_FQ01.PV")

# 生成时间特征（使用当前时间）
current_time = datetime.now()
input_data.update({
    "year": current_time.year,
    "month": current_time.month,
    "day": current_time.day,
    "weekday": current_time.weekday(),  # 0=Monday, 6=Sunday
    "hour": current_time.hour,
    "minute": current_time.minute,
    "second": current_time.second,
    "is_weekend": 1 if current_time.weekday() in [5, 6] else 0,
    "quarter": (current_time.month - 1) // 3 + 1
})

input_data

{'DQ200系统压力': 5.77,
 'AVS系统压力': 5.84,
 'DQ200累积流量': 333329696.0,
 'AVS累积流量': 55784552.0,
 'DQ200差分值': 1728,
 'AVS差分值': 996,
 'year': 2025,
 'month': 5,
 'day': 20,
 'weekday': 1,
 'hour': 10,
 'minute': 0,
 'second': 51,
 'is_weekend': 0,
 'quarter': 2}